In [4]:
from pathlib import Path
import re

import pandas as pd
from pymystem3 import Mystem
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

data_dir = Path("rutweetcorp/versions/3")

In [5]:
def load_tweets(path: Path, label: int) -> pd.DataFrame:
    frame = pd.read_csv(path, usecols=["ttext"], encoding="utf-8")
    frame = frame.rename(columns={"ttext": "text"})
    frame["positive"] = label
    return frame

positive = load_tweets(data_dir / "positive.csv", 1)
negative = load_tweets(data_dir / "negative.csv", 0)
data = pd.concat([positive, negative], ignore_index=True)
data = data.dropna(subset=["text"]).copy()
data["text"] = data["text"].astype(str)
data = data.loc[data["text"].str.strip().ne("")].reset_index(drop=True)

print("Всего твитов:", len(data))
print("Распределение классов:")
counts = data["positive"].value_counts().sort_index()
counts.index = counts.index.map({0: "negative (0)", 1: "positive (1)"})
counts.index.name = "sentiment"
print(counts)

examples = pd.concat([
    data.loc[data["positive"].eq(0)].head(3),
    data.loc[data["positive"].eq(1)].head(3),
], ignore_index=True)
examples["sentiment"] = examples["positive"].map({0: "negative", 1: "positive"})
examples[["text", "sentiment", "positive"]]

Всего твитов: 226834
Распределение классов:
sentiment
negative (0)    111923
positive (1)    114911
Name: count, dtype: int64


,text,sentiment,positive
0,на работе был полный пиддес :| и так каждое за...,negative,0
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",negative,0
2,@elina_4post как говорят обещаного три года жд...,negative,0
3,"@first_timee хоть я и школота, но поверь, у на...",positive,1
4,"Да, все-таки он немного похож на него. Но мой ...",positive,1
5,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,positive,1


In [6]:
def clean_text(text: str) -> str:
    return " ".join(re.findall(r"[а-яё]+", text.lower()))

def lemmatize_in_batches(texts: pd.Series, batch_size: int = 1000) -> pd.Series:
    mystem = Mystem()
    result = []
    try:
        for start in range(0, len(texts), batch_size):
            batch = texts.iloc[start:start + batch_size].tolist()
            analyzed = "".join(mystem.lemmatize(" | ".join(batch)))
            lemmatized = analyzed.split("|")
            result.extend(" ".join(text.split()) for text in lemmatized)
            if (start + len(batch)) % 20000 < batch_size or start + len(batch) == len(texts):
                print(f"Лемматизировано: {start + len(batch):,} / {len(texts):,}")
    finally:
        mystem.close()
    return pd.Series(result, index=texts.index, name="lemm_text")

data["clean_text"] = data["text"].map(clean_text)
data = data.loc[data["clean_text"].ne("")].copy()
data["lemm_text"] = lemmatize_in_batches(data["clean_text"])
data = data.loc[data["lemm_text"].ne("")].reset_index(drop=True)
lemm_examples = pd.concat([
    data.loc[data["positive"].eq(0)].head(3),
    data.loc[data["positive"].eq(1)].head(3),
], ignore_index=True)
lemm_examples["sentiment"] = lemm_examples["positive"].map({0: "negative", 1: "positive"})
lemm_examples[["text", "lemm_text", "sentiment", "positive"]]

Лемматизировано: 20,000 / 226,830
Лемматизировано: 40,000 / 226,830
Лемматизировано: 60,000 / 226,830
Лемматизировано: 80,000 / 226,830
Лемматизировано: 100,000 / 226,830
Лемматизировано: 120,000 / 226,830
Лемматизировано: 140,000 / 226,830
Лемматизировано: 160,000 / 226,830
Лемматизировано: 180,000 / 226,830
Лемматизировано: 200,000 / 226,830
Лемматизировано: 220,000 / 226,830
Лемматизировано: 226,830 / 226,830


,text,lemm_text,sentiment,positive
0,на работе был полный пиддес :| и так каждое за...,на работа быть полный пиддеса и так каждый зак...,negative,0
1,"Коллеги сидят рубятся в Urban terror, а я из-з...",коллега сидеть рубиться в а я из за долбать ви...,negative,0
2,@elina_4post как говорят обещаного три года жд...,как говорить обещаной три год ждать,negative,0
3,"@first_timee хоть я и школота, но поверь, у на...",хоть я и школоть но поверять у мы то же самый ...,positive,1
4,"Да, все-таки он немного похож на него. Но мой ...",да весь таки он немного похожий на он но мой м...,positive,1
5,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,ну ты идиотка я испугаться за ты,positive,1


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    data["lemm_text"],
    data["positive"],
    test_size=0.2,
    stratify=data["positive"],
    random_state=42,
)
print(f"Обучающая выборка: {len(X_train):,}; тестовая: {len(X_test):,}")

Обучающая выборка: 181,464; тестовая: 45,366


In [8]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_features=100_000,
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print("Размер обучающей TF-IDF матрицы:", X_train_tfidf.shape)
print("Размер тестовой TF-IDF матрицы:", X_test_tfidf.shape)

Размер обучающей TF-IDF матрицы: (181464, 100000)
Размер тестовой TF-IDF матрицы: (45366, 100000)


In [9]:
model = LogisticRegression(max_iter=1000, solver="liblinear", random_state=42)
model.fit(X_train_tfidf, y_train)

prediction = model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, prediction)
print(f"Accuracy на тестовой выборке: {accuracy:.4f}")

results = pd.DataFrame({
    "text": data.loc[X_test.index, "text"].values,
    "true_label": y_test.values,
    "predicted_label": prediction,
})
results.head(10)

Accuracy на тестовой выборке: 0.7523


,text,true_label,predicted_label
0,RT @hl0pec: @Oleska_Sha @dasponomarenko @PrVla...,1,1
1,А у меня новый год:-) #рабочееместо #работа #п...,1,1
2,Пора домой! Мне сказали... все пошли домой... ...,0,0
3,"Я в трауре, после каникул мама запретит есть с...",0,0
4,свой полу тысячный твит посвящаю музыке!!! без...,1,1
5,Второй день снится измена( почему я к снам так...,0,0
6,"Шлите ваши фото морозной Калуги, будем смотрет...",1,1
7,"@Guynel0495 ну да,там же наверно актау-.- тако...",1,1
8,@sempi_pony @___AfuckingA___ @HappyNewDeath на...,0,0
9,“@skarlett1965: @belogolovcev Да! Но.. она не ...,0,0
